In [7]:
# Pipeline parameters

SourceId = 1
SourceName = "SYNTHEA"
SourceEntity = "patients"
SourcePath = "Files/synthea/extracted/clinical/patients.csv"
SourceFormat = "CSV"
TargetTable = "bronze_patients"
LoadType = "Full"
WatermarkColumn = ""
LastWatermark = ""
PipelineRunId = "XM10101"


StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 9, Finished, Available, Finished, False)

In [8]:
# # Pipeline parameters
# SourceId = 0
# SourceName = ""
# SourceEntity = ""
# SourcePath = ""
# SourceFormat = ""
# TargetTable = ""
# LoadType = ""
# WatermarkColumn = ""
# LastWatermark = ""
# PipelineRunId = ""

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 10, Finished, Available, Finished, False)

In [9]:
# Import libraries
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

from pyspark.sql import functions as F
from datetime import datetime
import json

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 11, Finished, Available, Finished, False)

#### Show incoming configuration

In [10]:
print(f"SourceId        : {SourceId}")
print(f"SourceName      : {SourceName}")
print(f"SourceEntity    : {SourceEntity}")
print(f"SourcePath      : {SourcePath}")
print(f"SourceFormat    : {SourceFormat}")
print(f"TargetTable     : {TargetTable}")
print(f"LoadType        : {LoadType}")
print(f"WatermarkColumn : {WatermarkColumn}")
print(f"LastWatermark   : {LastWatermark}")
print(f"PipelineRunId   : {PipelineRunId}")

TargetLakehouse = "HP_01_LH_Health_Bronze"
TargetSchema = "bronze"

TargetFullTableName = (
    f"{TargetLakehouse}.{TargetSchema}.{TargetTable}"
)


StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 12, Finished, Available, Finished, False)

SourceId        : 14
SourceName      : CMS
SourceEntity    : hospitals
SourcePath      : Files/synthea/extracted/external/hospitals.csv
SourceFormat    : CSV
TargetTable     : bronze_cms_hospitals
LoadType        : Full
WatermarkColumn : 
LastWatermark   : 
PipelineRunId   : XM10101


#### Normalize the values

In [11]:
# ============================================================
# NORMALIZE PIPELINE PARAMETERS
# ============================================================

load_type = (LoadType or "").strip().upper()
source_format = (SourceFormat or "").strip().lower()

# Normalize watermark column
watermark_column = (
    str(WatermarkColumn).strip()
    if WatermarkColumn is not None
    and str(WatermarkColumn).strip().upper() not in ["", "NULL"]
    else None
)

# Normalize previous watermark
last_watermark = (
    str(LastWatermark).strip()
    if LastWatermark is not None
    and str(LastWatermark).strip().upper() not in ["", "NULL"]
    else None
)

print("========================================")
print("NORMALIZED PARAMETERS")
print("========================================")
print("Load Type:", load_type)
print("Source Format:", source_format)
print("Watermark Column:", watermark_column)
print("Previous Watermark:", last_watermark)
print("========================================")


StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 13, Finished, Available, Finished, False)

NORMALIZED PARAMETERS
Load Type: FULL
Source Format: csv
Watermark Column: None
Previous Watermark: None


#### Read the source file

In [12]:
# ============================================================
# READ SOURCE
# ============================================================

print("Source Path:", SourcePath)
print("Source Format:", source_format)

if source_format == "csv":

    df_source = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(SourcePath)
    )

elif source_format == "parquet":

    df_source = spark.read.parquet(SourcePath)

else:
    raise ValueError(
        f"Unsupported SourceFormat: '{source_format}'"
    )


# ============================================================
# CLEAN COLUMN NAMES FOR DELTA
# ============================================================

import re

def clean_column_name(name):
    name = name.strip()
    name = re.sub(r"[^A-Za-z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

df_source = df_source.toDF(
    *[clean_column_name(c) for c in df_source.columns]
)

print(f"Successfully read: {SourceEntity}")
print(f"Path: {SourcePath}")
print(f"Columns: {len(df_source.columns)}")

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 14, Finished, Available, Finished, False)

Source Path: Files/synthea/extracted/external/hospitals.csv
Source Format: csv
Successfully read: hospitals
Path: Files/synthea/extracted/external/hospitals.csv
Columns: 40


#### Add ingestion metadata

In [13]:

# ============================================================
# ADD METADATA
# ============================================================
df_source = (
    df_source
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit(SourceName)
    )
    .withColumn(
        "_source_entity",
        F.lit(SourceEntity)
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(PipelineRunId)
    )
)


# ============================================================
# VALIDATE
# ============================================================

rows_read = df_source.count()

print(f"Rows read: {rows_read:,}")

df_source.printSchema()


StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 15, Finished, Available, Finished, False)

Rows read: 5,419
root
 |-- Facility_ID: double (nullable = true)
 |-- Facility_Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City_Town: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP_Code: integer (nullable = true)
 |-- County_Parish: string (nullable = true)
 |-- Telephone_Number: string (nullable = true)
 |-- Hospital_Type: string (nullable = true)
 |-- Hospital_Ownership: string (nullable = true)
 |-- Emergency_Services: string (nullable = true)
 |-- Meets_criteria_for_birthing_friendly_designation: string (nullable = true)
 |-- Hospital_overall_rating: string (nullable = true)
 |-- Hospital_overall_rating_footnote: string (nullable = true)
 |-- MORT_Group_Measure_Count: string (nullable = true)
 |-- Count_of_Facility_MORT_Measures: string (nullable = true)
 |-- Count_of_MORT_Measures_Better: string (nullable = true)
 |-- Count_of_MORT_Measures_No_Different: string (nullable = true)
 |-- Count_of_MORT_Measures_Worse: string (nulla

#### FULL load

In [14]:
new_watermark = None

if load_type == "FULL":

    print(f"FULL load: {SourceEntity}")

    rows_written = df_source.count()

    target_table = f"bronze.{TargetTable}"

    print(f"Writing to Bronze table:{TargetTable}")

    (
        df_source.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"Rows written: {rows_written:,}")

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 16, Finished, Available, Finished, False)

FULL load: hospitals
Writing to Bronze table:bronze_cms_hospitals
Rows written: 5,419


#### INCREMENTAL load

In [15]:
if load_type == "INCREMENTAL":

    if not watermark_column:
        raise ValueError(
            f"WatermarkColumn is required for incremental load: {SourceEntity}"
        )

    print(f"INCREMENTAL load: {SourceEntity}")
    print(f"Watermark column: {watermark_column}")
    print(f"Previous watermark: {last_watermark}")

    df_incremental = df_source.withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(watermark_column))
    )

    if last_watermark is None:
        print("No previous watermark - processing all available rows.")
    else:
        df_incremental = df_incremental.filter(
            F.col("_watermark_ts") >
            F.to_timestamp(F.lit(last_watermark))
        )

    watermark_result = (
        df_incremental
        .agg(
            F.max("_watermark_ts").alias("NewWatermark")
        )
        .first()
    )

    new_watermark = watermark_result["NewWatermark"]

    df_incremental = df_incremental.drop("_watermark_ts")

    rows_written = df_incremental.count()

    print(f"Rows to write: {rows_written:,}")

    target_table = f"bronze.{TargetTable}"

    print(f"Writing to Bronze table: {target_table}")

    if rows_written > 0:

        (
            df_incremental.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

        print(f"Rows written: {rows_written:,}")
        print(f"New watermark: {new_watermark}")

    else:
        print("No new rows found.")

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 17, Finished, Available, Finished, False)

In [16]:
import json

result = {
    "RowsRead": int(rows_read),
    "RowsWritten": int(rows_written),
    "RowsRejected": 0
}

notebookutils.notebook.exit(json.dumps(result))

StatementMeta(, 8e89ec57-658d-468d-b6d7-e3bc4f2f946c, 18, Finished, Available, Finished, False)

ExitValue: {"RowsRead": 5419, "RowsWritten": 5419, "RowsRejected": 0}